hello everyone my name is Jonathan and today I'm going to be taking you throughthis stable diffusion Deep dive notebook looking into the code behind the kind of popularhigh-level apis and libraries and tools and so on to see what exactly does thegeneration process look like and how can we modify that how do each of theindividual components work so feel free to run along with me if you haven't beforethis might take a little while to run just because it's downloading these large modelsif they aren't already downloaded and loading them up and so we're going to startby just kind of recreating what it looks like to generate an image using sayone of the existing pipelines in hugging face so we're going to basically have copiedthe code from the core method of the default stable diffusion Pipeline and so ifyou go and view that here you'll see that we're going to be basically replicatingthis code but now we'll be doing it on our own sort of notebook andthen we'll slowly understand what each of these different parts is doing so we've gotsome setup we've got some sort of loop running through a number of sampling timesteps and regenerating an image so this is supposed to be a watercolor picture ofan otter and it's very very cool that this model can just do this butnow we want to know how does that actually work what's going on so thefirst component is the autoencoder now this stable diffusion is a latent diffusion model andwhat that means is that it doesn't operate on pixels it operates on go inthe latent space of some other autoencoder model in this case a variational autoencoder that'sbeen trained on a large number of images to compress them down into this latentrepresentation and bring them back up again so I have some functions to do thatwe're going to look at what it's like in action by just downloading a picturefrom the internet opening it up with pil so we have this 512 by 512pixel image and we're going to load it in and then we're going to useour function defined above to encode that into some latent representation and what this isdoing is calling the vae.encode method on a tensor version of the image and thatgives us a distribution and so be sampling from that distribution and we're scaling bythis because that's what the authors of the the model did they scaled the latentsdown before they fit them to the model and so we have to do thatscaling and then the reverse when we're decoding just to be consistent with that butthe key idea is that we go from this big image down to this 4x64by 64 latent representation right so we've gone from this much larger image down andif we visualize what the four channels here and this four different 64 by 64channels what that looks like we'll see that it's capturing something of the image rightyou can sort of see the same shapes and things there but it's not quitea direct mapping or anything for example there's this weirdness going on the beak someof the channels look slightly Stranger than the others so there's some sort of Richinformation captured there and if we decode this back what we'll see is that thedecoded image looks really good you really have to look closely to tell the differencebetween our input image here and the decoded version so very very impressive compression rightthis is a factor of eight in each Dimension so 512 by 512 down 64by 64. it's like a factor of 64 reduction in in data but it's still somehow capturingmost of that information it's a very information Rich representation and this is going tobe great because now we can work with that with our diffusion model and getnice high resolution results even though we're only working with these 64 by 64 latentsnow it doesn't have to be 64 by 64.you can go and modify this to say what if this is you know 640and encode that down and you'll see that it's just that same factor of 8reduction um and there we go now we have 80 by 64.and this just has to be a multiple of eight otherwise you'll get I thinkan error okay so we have our encoded version of this image and that's prettygreat the next component we're going to look at is the scheduler and I'll lookmore closely at this later but for now we're going to focus on this ideaof adding noise right so during training we add some noise to an image andthen the model tries to predict what that noise is and we're going to dothat to different amounts so here we're going to recreate the the same type ofschedule and you can try different schedulers from the library oops and these parameters hereBeta start beta end beta schedule that's how much noise was added at different timesteps and how many time steps are used during training for sampling we don't wantto have to do a thousand steps so we can set a new number oftime steps and then we'll see how these correspond with the scheduler.time steps attribute tothe original um training time steps so here we're going to have 15 sampling stepsand that's going to be equivalent to starting at times at 909 and just movinglinearly down to time steps zero we can also look at the actual amount ofnoise present with the sigmas attribute so again starting High moving down and if youwant to see what that schedule looks like we can plot that here and ifyou want to see the time steps you'll see that it's just a linear relationshipso there we go we're going to start at a very high noise value andwe're going to slowly slowly try and reduce this down until ideally we get animage out okay so this Sigma is the amount of noise added let's see whatthat looks like so I'm going to start with some random noise that's the sameshape as my latent representation my encoded image and then I'd like to be equivalentto sampling step 10 out of 15 here so I'm going to go and lookup what time set that equates to and that's going to be one of thearguments that I passed the scheduler.add noise function so I'm calling scheduler.add noise giving itmy encoded image the noise and what time step I'd like to be noising equivalentto and this is going to give me this noisy but still recognizable version ofthe image and you can go and say okay what if I look at somewhereearlier in the process right does it look more noisy what about right at thebeginning right at the end and feel free to play around there um okay sothis adding noise what are we actually doing what does the code look like let'sinspect the function and you'll see that there's some setup for different types of argumentand shapes but the key line is just this noisy samples is equal to originalsamples plus the noise scaled by the sigma parameter right so that's all it isit's not always the same different papers and implementations will add the noise slightly differentlybut in this case that's all it's doing so scheduler.add noise just adding noise that'sthe same shape as the latents scaled by the sigma parameter um okay so that'swhat we're doing um so if we want to start from random noise instead ofa noisy image we're going to scale it by that same Sigma value so thatit looks the same as an image that's been scaled by that amount but thenbefore we feed that to the actual model we then have to handle that scalingagain you could do it like this but now we have this scale model inputfunction associated with the scheduler just to hide that complexity away okay so now we'regoing to look at the same kind of sampling loop as before but we're goingto start now with our image we're going to take our encoded image we're goingto noise it to some time step and then we only can denoise from thereso in code we are now preparing our text and everything the same as beforewhich we'll look at we're setting our number of inference steps to 50 right I'minference steps is equal to 50 here and we're saying I'd like to start atthe equivalent of Step 10 out of 50.so I'll look up what time step that equates to I'll add noise to myimage equivalent to that step and then we're going to run through sampling but thistime we're only going to start doing things once we get above that start stepso I'm going to ignore the first 10 out of 50 steps and then beyondthat I'm now going to start with this noisy version of my input image andI'm going to denoise it according to this prompt and the Hope here is thatby starting from something that has some of the sort of rough structure and colorof that input image I can kind of fix that into my generation but I'vegot a new prompt a National Geographic photo of a colorful dancer and here wego we see this is the same sort of thing as the parrots but nowwe have this completely different actual content thanks to a different prompt and so that'sa fun kind of use of this image to image process you might have seenthis for taking drawings adding a bunch of noise and then denoising them into fancypaintings and so on so again this is something that there's existing tools for thisright the strength parameter and the image to image pipeline that's just something like thisum what step are we starting at how many steps are we skipping um butyou can see that this is a pretty powerful technique for getting a bit ofextra control over like composition and color and a bit of the structure um okayso that's that's that trick with adding noise and then using that as image toimage the next big section I'd like to look at is how do we gofrom a piece of text that describes what we want into a numerical representation thatwe can feed to the model so we're going to trace out that Pipeline andalong the way we'll see how we can modify that for a bit of funso step number one we're taking our prompt and we're turning it into a sequenceof discrete tokens so here we have in this case 77 because that's the maximumlength um discrete tokens it's always going to be that if your prompt is longerit'll truncate it and if we decode these tokens back we'll see that we havea special token for the start of the text then a picture of a puppyand then the rest is all the same token which is this kind of endof text padding token right so we have this special token for puppy this specialtoken has its own meaning end of text and the the prompts are always goingto be padded to be the same length so before in the code that wewere using there we always jump straight to the so-called output embeddings which is whatwe fed to the model as conditioning and so somehow this captures some information aboutthis prompt and but now we want to say well how do we get therehow do we get from this sequence of tokens to these output embeddings what isthis text encoder um forward pass doing right so we can look at this andthere's going to be multiple steps the first is going to be some embeddings soif we look at the text encoder.txt model.embeddings we'll see there's a couple of differentones we have token embeddings right and so this is to take those individual tokenstoken 49408 or whatever and map it into a representation that's a numerical representation sohere it's a London bedding there are seven sorry about 50 000 rows one foreach token and for each token we have 768 values so that's the embedding ofthat token and if we want to feed one in and see what the embeddinglooks like here's the token for puppy and here's the token embedding right 768 numbersthey somehow capture that meaning of that token on its own and we can dothe same for all of the tokens in our prompt so we feed them throughthis token embedding layer and now we get 77 768 dimensional representations of this ofeach token now these are all on their own no matter where in the sentenceis it is the token embedding will be the same so the next step isto add some positional information some models will do this with some kind of likelearned pattern of positioning but in this case the positional embedding is just another Londonbedding but now instead of having one embedding for every token we have one embeddingfor every position out of all 77 possible positions and so just like we didfor the tokens we can feed in the position IDs one for every possible positionand we'll get back out an embedding for every position in the prompt um combiningthem together there's again multiple ways people do this in the literature but in thiscase it's as simple as adding them that's why they made them the same shapeso that you can just add the two together and now these input embeddings havesome information related to the token and some related to the position so so farwe haven't seen any big model just two London beddings but this is getting everythingready to feed through that model and so we can check that this is thesame as if we just called the embeddings layer of that model which is goingto do both of those steps at once but we'll see just now why wewant to separate that out into individual ones okay so we have these individual tokensand they have some positional information we have these final embeddings now we'd like toturn them into something that has a richer representation thanks to some big Transformer modeland so we're going to feed these through and I made this little diagram hereeach token is going to turn into a token embedding get combined with the positionalembedding and then it's going to get fed through this Transformer encoder which is justa stack of these blocks and so each block has some magic like attention hassome feed forward components there's additions and normalizations and skips and so on as wellbut we're going to have some number of these blocks all stacked together and theoutputs of each one get fed into the next block and so on until weget our final set of hidden States these encoder hidden States aka the output embeddingsand this is what we feed to our units to make its predictions so theway we get this I just copied the text encoder.textmodel.forward method pulled out the relevantbits we are going to take in those input embeddings combined positional and token embeddingsand we're going to feed that through the text model.encoder function with some additional parametersaround attention masking and telling it that we'd like to Output the hidden States ratherthan the final outputs so if we run this we can just double check theseembeddings are going to look just like the output embeddings we saw right at thebeginning so we've taken that one step tokens to output embeddings and we've broken itdown into this number of smaller steps where we have tokenization getting our token embeddingscombining with position embeddings feeding it through the model and then that gives us thosefinal outputs so why have we gone through this trouble well there's a couple ofthings we can do one demoed here I'm getting the token embeddings but then I'mlooking up um where is the token for puppy and I'm going to replace itwith a new set of embeddings and this is going to be another just learnedembedding of this particular token here two three six eight right so I'm kind ofcutting out the token embedding for puppy slipping in this new set of token embeddingsand I'm going to get some output embeddings which at the start look very similarto the previous ones in fact identical but as soon as you get past theposition of puppy in that prompt you're going to see that the rest have changedright so we've somehow messed with these embeddings by slipping in this new token embeddingright at the start and if we generate with those embeddings which is what thisfunction is doing we should see something other than a puppy and sure enough drumrolluh we don't we get a cat and so now you know what token twothree two three six eight means um we've managed to slip in a new tokenembedding and get a different image okay what can we do with this why isthis fun um well a couple of Tricks first off we could look up thetoken embedding for skunk right which is this number here and then instead of nowjust replacing that in place of puppy what if I make a new token embeddingthat's some combination of the embedding of puppy and the embedding of skunk right soI'm taking these two token embeddings I'm just averaging them and I'm inserting them intomy set of token embeddings for my prompt um in place of just the wordpuppy and so hopefully when we generate with this we get something that looks abit like a puppy a bit like a skunk and this doesn't work all thetime but it's pretty cute when it does there we go puppy skunk hybrid okayso that's not the real reason we're looking at this the main application at themoment of being able to mess with these token embeddings is to be able todo something called textual inversion so in textual inversion we're going to have our prompttokenize it and so on and but here we're going to have a special learnedembedding for some New Concept right and so the way that's trained is going tobe outside of the scope of this notebook but there's a good blog post andCommunity notebooks and things for doing that but let's just see this in application hereso I've um there's a whole library of these Concepts stable diffusion concept Library whereyou can browse through um tons and tons and tons look over 1 400 differentCommunity contributed token embeddings that people have trained and so I'm going to use thisone here this verb style here's some example outputs and then these are the imagesit was trained on so these pretty little bird paintings done by my mother I'vetrained a new token embedding they tries to capture the essence of the style andthat's represented here in this learned embeds.bin so if you download this and then uploadit to wherever your notebook's running I have it here londonbeds.bin we can load thatin and you'll see that it's just a dictionary where we have one key that'sthe name of my new style and then we have this token embedding 768 numbersand so now instead of slipping in the token embedding for cat we're going toslip in this new embedding which we've loaded from the file into this prompt somasking the style of puppy tokenize get my token embeddings and then I'm going toslip in this replacement embedding um in place of the embedding for puppy and whenwe generate with that we should hopefully get a mouse in the style of thiskind of cutesy watercolor on rough paper image and sure enough that's what we getvery cute little drawing of a mouse in an apron apparently uh okay so veryvery cool application again there's a nice inference notebook that makes this really easy youcan say a katoya in the style of verb style you don't have to worryabout manually replacing the token embeddings yourself but it's good to know what the codelooks like under the hood right how are we doing that what stage of thetext embedding process we're modifying very fun to get a bit of extra control anda very useful technique because now we can kind of augment our model's vocabulary withouthaving to actually retrain the model itself we're just learning a new token embedding sovery very powerful idea and really fun to play with and like I said there'sthousands of community contribute contributed tokens but you can also train your own I thinkI link The Notebook from here but it's also in all the docs and soon here's the training notebook okay final little trick with embeddings rather than messing withthem at the token embedding level we can push the whole prompt through that entireprocess to get our final output embeddings and we can mess with those at thatstage as well so here I have two prompts a mouse and a leopard tokenizingthem encoding them with the text encoder so that's that whole process and these finaloutput embeddings I'm just going to mix them together According to some factor and generatewith that and so you can try this with you know a cat and asnake um and you should be able to get some really fun uh different chimerasand oops a snail apparently okay well I can't spell um but yeah have funwith that doesn't have to be animals I'd love to see what you create withthese weird mixed up Generations okay we should look at the actual model itself thekey unit model the diffusion model um what is it doing what is it predictingwhat is it accepting as arguments so this is the kind of cool signature wecall our units forward pass and we feed in our noisy latents the time stepand it's like the training time step and the encoder hidden States right so thosetext embeddings that we've just been having fun with so doing that without any Loopsor anything I'm sitting in my scheduler getting my time step getting my noisy latentand my text embeddings and then we're going to get our model prediction and we'lllook at the shape of that and you'll see that this prediction has the sameshape as the latents and given these noisy latents what the model is predicting isthe noise component of that and actually it's predicting the noise component scaled by Sigmaso if we wanted to see what the original image looks like we could saywell the the denoise latents is going to be the current noisy latents minus Sigmatimes the model prediction right um and so when we're denoising we're not going togo straight to that upward prediction we're going to just remove a little bit ofthe noise at a time but it might be useful to visualize what that finalprediction looks like so that's what we're doing here making a folder to store someimages preparing our text scheduler and input and then we're going to do this Loopbut now we're going to get the model prediction and instead of just updating ourlatency by one step we're also going to store an image all right I'm decodingthese two images an image of the predicted completely denoised like original sample so that'sthis predicted original sample here you could also calculate this yourself latent 6 error isequal to the current latents minus Sigma times the noise prediction right so those twoshould work equivalently but this Loop is going to run and it's going to savethose images to the steps folder which we can then visualize and so once thisfinishes in a second or two on the left we're going to see the kindof noisy input to the model at each stage and on the right we're goingto see the noisy inputs minus the noise prediction right so the denoised version andso just give it a second or two to run it's taking it a littlebit longer because it's decoding those images each time saving them um but once thisfinishes we should have a nice little preview video okay here we go so thisis the noisy latent and if we take the model's noise prediction and subtract itfrom that we get this very blurry output and so you'll see as we playthis oh I've left some modifications in from last time sorry um when you seethis this guidance scale we'll be back at uh I think it was eight umin the next section we'll talk about classified free guidance and so I've been modifyingthat example my bad I might cut this out of the video we'll see soI've got to wait a few seconds again for that to generate and I'll doso as patiently as I can okay so here we go again the noisy inputsthe predicted denoise version and you can see at the start it's very blurry butover time it gradually converges on our final output um and you'll notice that onthe left these are the latents as they are each step they don't change particularlydrastically a little bit at a time but at the start when the model doesn'thave much to go on its predictions do change quite a bit at each stepright it's much less well defined and then as we go forward in time itgets more and more refined better and better predictions and so it's got a amore accurate estimation of the noise to remove and we remove that noise gradually untilwe finally get our output quite fun to visualize the process and hopefully that helpsyou understand why we don't just make one prediction and do it in one stepright because we get this very blurry mess but instead we do this kind ofiterative um sampling there which we'll talk about very shortly before then though the finalthing I should mention classify free guidance what is that well like you saw whenI accidentally generated the version with a much lower guidance scale um the way classifierfree guidance works is that in all of these Loops we haven't actually been passingone set of noisy latents through the model we've been passing two identical versions andas our text embeddings we've not just been passing the embeddings of our prompts rightthese ones here we've been concatenating them with some unconditional embeddings as well and whatthe unconditional embeddings are is just a blank prompt right no text whatsoever so justall padding passing that through so when we get our predictions here we've given intwo sets of latents and two sets of text embeddings we're gonna get out twopredictions for the noise so we'll be splitting that apart one prediction for the unconditionallike no prompt version and one for the prediction based on the prompt and sowhat we can do now is we can save all my final prediction is goingto be the unconditional version plus the guidance scale times the difference right so ifyou think about it if I predict without the noise I'm predicting here if Ipredict with the noise sorry with the text encoding with the prompt I get thisprediction instead and I'd like to move more in that direction I'd like to pushit even further towards the prompt version and and Beyond and so this guidance scalecan be larger than one to push it even more in that direction and thisit turns out is kind of key for getting it to follow the prompt nicelyand I think it was first um brought up in the Glide paper AI coffeebreak on YouTube has a great video on that but yeah really useful trick orreally neat hack depending on who you talk to but it does seem to workand the higher the guidance scale the more the model will try and look likethe prompts kind of in the extreme versus the lower guidance scale it might justtry and look like a generic good picture okay we've been hiding away some complexityin terms of this scheduler.step function so I think we're going to step away fromThe Notebook now and scribble a bit on some paper to try and explain exactlywhat's going on with sampling and so on and then we'll come back to thenotebook for one final trick all right so here's my take on sampling and tostart with I'd like you to imagine the space of all possible images so thisis a very large High dimensional space for 256 by 256 by 3 image thatis 200 000 dimensional and my paper unfortunately is only two dimensional so we're goingto have to squish this down a fair bit and use our imagination now ifyou just look at a random point in this space this is most likely notgoing to look like anything recognizable it'll probably just look like gobbled noise but ifwe map an image into this space um we'll see that it has some sortof fixed point and a very similar image almost pixel equivalent it's going to bevery close by now there's this theory that you'll hear talked about called manifold Theorywhich says that for most real images like a data set of images these aregoing to lie on some lower dimensional manifold within this higher dimensional space right inother words if we map a whole bunch of images into the space they're notgoing to fill the whole Space they're going to be kind of clustered onto somesurface now I've drawn it as a line here because we stuck with 2D butthis is a much higher dimensional plane equivalent okay so each of these ones hereis some image and the reason that I'm starting with this is because we'd liketo generate images we'd like to generate plausible looking images not just random nonsense andso we'd like to do that with diffusion models so where did they come inwell we can start with some image here some real image from our training dataand we can push the way from the manifold of like plausible existing images bycorrupting it somehow so for example just adding random noise that's equivalent to like movingin some random Direction in this space of all possible images and so that's goingto push the image away and then we can try and predict using some modelwhat this noise looks like right how do I go from here back to aclosable image what is this noise that's been added and so that's going to beour our big unit that does that prediction that's going to be our diffusion modelgreat and so that's um in in this language going to be called something likea score function right how do I get from wherever I am what's the noisethat I need to remove to get back to a plausible image um okay sothat's all well and good um we can train this model with a number ofexamples because we can just take our training data add some random noise predict predicttry and predict the noise update on model parameters so we can hopefully learn thatfunction fairly well now we'd like to generate with this model right so how dowe do that well we can start at some random point right like let's let'sstart over here um and you might think well surely I can just now predictthe noise remove that and then I get my output image and that's great exceptthat you've got to remember now we're starting from a random point in the spaceof all possible images it just looks like garbled nonsense and the model's trying tosay well what does the noise look like and so you can imagine here fortraining the further away we get from our examples the sparse of our training willhave been um but also it's not like it's very obvious how we got tothis noisy version right we could have come from this image over here added abunch of noise we could have come from one over here one over here andso this model is not going to be able to make a perfect prediction atbest it might say well somewhere in that direction right it could Point towards somethinglike the data set mean or at least the edge that's closer but it's notgoing to be able to perfectly give you one nice solution and sure enough that'swhat we see if we sample diffusion models just in one step we get thepredictions look at what that corresponds to as an image it's just going to looklike a blurry mess maybe like the mean of the data or you know somesort of garbled output definitely not going to look like a nice image so howdo we do better and the idea of sampling is to say well there's acouple of framings so I'll start with the existing framing that you'll see talked abouta lot of score based models and so on and then we'll talk about someother ways to think about it as well so this process of gradually corrupting ourimages away adding a little bit of noise at a time people like to talkof this as a stochastic differential equation stochastic because there's some Randomness right we're pickingrandom amounts of noise random directions to add and a differential equation because it's nottalking about anything absolute just how we should change this from moment to moment toget more and more corrupted right so that's why it's a differential equation and withthat framing the question of well how do I go now back to the imagethat's framed as solving an ordinary differential equation that corresponds to like the reverse ofthis process now you can't solve Odes in a single step but you can findan approximate solution um and the more sort of sub steps you take the betteryour approximation and so that's what these samples are doing given like okay we saidthis image over here here's my prediction rather than moving the whole way there inone go we'll remove some of that noise right do a little updates and thenwe'll get a new prediction right and so maybe now the prediction is slightly betterit says up here so we move a little bit in that direction and nowit makes an even better prediction because as we get closer to the manifold rightas we have less and less noise and more and more of like some imageemerging the model is able to get more more accurate predictions and so in somesort of number of steps we divide up this this process and we get closerand closer and closer until we ideally find some image that looks very plausible asour output and so that's what we're doing here with a lot of these samplesthey're effectively trying to solve this ode in some number of steps by um yeahbreaking the process up and only moving a small amount at a time now youget sort of first order solvers right where all we're doing is just linearly movingwithin each one and this is equivalent to something called Euler's method or Euler's methodif you're like me and you've only ever read it and this is what someof the most basic Samplers are doing just linear approximations for each of these littlesteps um but you also get additional approaches so for example um maybe if wewere to make a prediction from here it might look like something like this andif we were to make a prediction from here it might look like something likethat so we have our error here but as we move in that direction it'salso changing right so there's like a derivative of a derivative a gradient of agradient and that's where this kind of second order solver comes in and says wellif I know how this prediction changes as I move in this direction like whatis the derivative of it then I can kind of account for that curvature whenI make my update step and maybe know that it's going to curve a bitin that direction and so that's where we get things like these so-called second ordersolvers and higher order solvers the upside of this is that we can get youknow do a larger step at a time because we have a more accurate predictionwe're not just doing a first order linear approximation we have this kind of curvaturetaken into account the downside is that to estimate that curvature for a given pointwe might need to call our model multiple times to get multiple estimates and sothat takes time so we can take a larger step but we need more modelevaluations per step a kind of hybrid approach is to say well rather than tryingto estimate the curvature here I might just take a linear step look at thenext prediction but I'll keep a history of my previous steps and so then overhere it predicts like this so I have now this history and I'm going touse that to better guess what this trajectory is so I might keep a historyof the past you know three or four or five predictions and know that sincethey're quite close to each other maybe that tells me some information about the curvaturehere and I can use that to again take larger steps and so that's whatwe see the circle you know linear multi-step sampling coming in just keeping this bufferof past predictions to try and do a better job estimating than the simple youknow one step linear type first order solvers okay so that's the school-based um samplingversion and all of the variants and Innovation comes down to things like how canwe do this in as few steps as possible you know maybe we have aschedule that says we take larger steps at first and then gradually smaller steps aswe get closer um you know there's I think now some Dynamic methods and canwe estimate how many steps we need to take and so on um so that'sall trying to attack it from this kind of score based ode solving framework butthere's another way to think of this as well and that's to say okay wellI don't really care about solving this exact reverse ode right all I care aboutis that I end up with an image that's on this manifold like a plausiblelooking image and so I have a model that estimates how much noise there isright and if that noise is very small then that means I've got a goodimage and if that noise is really large then that means I've got some workto do and so this kind of starts bringing up some analogies to training neuralnetworks because the neural networks we have the space of all possible parameters and we'retrying to adjust those parameters not to solve the gradient flow equation right although thatthat's you know possible in theory that you might try and do that we don'tcare about that we just want to find a Minima we want to find apoint where our loss is really good and so when we're training a neural networkthat's exactly what we do we set up an Optimizer we take some number ofsteps trying to reduce some loss and once that loss gets sort of you knowlevels off right reduce over time levels off okay cool I guess we found agood neural network um and so we can apply that same kind of thinking hereto say all right I'll start at some point and I have an estimate ofthe gradient right it's like maybe pointing over here but remember that estimate is notvery good just like the first gradients estimated when training in your network are prettybad because it's all just these randomly initialized weights but hopefully at least points ina useful Direction so then I'll take some step and the length of this stepI won't try and do some fancy schedule I'll just offload this to a sortof off-the-shelf Optimizer right so I have some learning rate maybe something like momentum thatdetermines how big of a step I take and then I update my prediction righttake another step in that direction and so on so now instead of following afixed schedule we can use tricks that have been developed for training neural networks rightadaptive learning rates momentum weight Decay and so on and we can apply them backto this kind of sampling case and so it turns out this works okay I'vetried this for stable diffusion needs some tricks to get it working but it's aslightly different way of thinking about sampling rather than relying on sort of a hard-codedode solver that you've figured out yourself just saying why don't we treat this likean optimization problem where if the model predicts almost no noise that's good we're doinga good job and if the model predicts lots of noise then we can usethat as a gradient and take a gradient update step according to our Optimizer andtry and sort of converge on a good image as our output and this thisis you know you can stop early once your model prediction is sufficiently low forthe amount of Noise Okay cool I'm done and so I found you know in10 15 Steps you can get some pretty good images out um yeah so that'sa different way of viewing it not so popular at the moment but maybe hopefullysomething we'll see um yeah just a different Framing and for me at least thathelps me think about what we're actually doing with these Samplers we're trying to finda point where the model predicts very little noise so starting from a bad predictionmoving towards it getting better by looking at this estimated amount of noise as oursort of gradient and solving that um just kind of iteratively removing bits at atime so I hope that helps elucidate the different kinds of Samplers and the goalof that whole thing and also illustrate at least why we don't just do thisin a single step right why we need some sort of iterative approach otherwise we'dend up with just very bad blurry predictions all right I hope that helps nowwe're going to head back to the notebook to talk about our final trick ofguidance okay the final part of this notebook guidance how do we add some extracontrol to this generation process right so we already have control via the text andwe've seen how we can modify those embeddings we have some control by starting ata noisy version of an input image rather than Pure Noise to kind of controlthe structure but what if there's something else what if we'd like a particular styleor to enforce that the model looks like some input image or maybe sticks tosome color palette um it would be nice to have some way to add thisadditional control and so the way we do this is to look at um someloss function on the decoded denoised predicted image right the predicted denoise meets final outputand use that loss to then update the noisy latents as we generate in adirection that tries to reduce that loss so for demo we're going to make avery simple loss function I would like the image to be quite blue and toenforce that my error is going to be the difference between the blue channel rightred green blue blue is the third channel of the color channels and the differencebetween the blue Channel and 0.9 so the closer all the blue values are to0.9 the lower my error will be so that's going to be my kind ofguidance loss and then during sampling here what I'm going to do everything is goingto be the same as before but um every few iterations you could do itevery iteration but that's a little slow so here every five iterations I'm going toset requires grad equals true on the latents I'm then going to compute my predicteddenoised version I'm going to decode that into image space and then I'm going tocalculate my loss using my special blue loss and scale it with some scaling Factorthen I'm going to use torch to find the gradient of this loss with respectto those latents those noisy latents and I'm going to modify them right and Iwant to reduce the loss I'm going to subtract here this gradient multiplied by Sigmasquared because we're going to be working at different noise levels and so if werun this we should see hopefully it's going to do that same sort of samplingprocess as before but we also are occasionally modifying our latents by looking at thegradient of the loss with respect to those latents and updating them in a directionthat reduces that loss and sure enough we get a very nice blue picture outand if I change the scale here down to something lower and run it we'llsee that um scale is lower so the loss is lower so our modifications tothe latents are smaller and we'll see that we get out a much less blueimage there we go so that's the default image very red and dark because it'sthe prompt is just a picture of a campfire but as soon as we addour um our additional loss or guidance we're going to get out something that bettermatches that additional constraint that we've imposed right so this is very useful not justfor making your images blue but like I said color palettes or using some classifiermodel to make it look like a specific class of image or using a modellike clip to again associate it with some text so lots and lots of differentthings you can do now a few things I should note one we decoding theimage back to image space calculating a loss and then tracing back that's very computationalLee intensive compared to just working in latent space and so we can do thaton every fifth operation to reduce the time but it still is much slower thanjust your generic sampling and then also um we're actually still cheating a little bithere because what we should do is set requires grad equals true on the latentsand then use those to make our noise prediction use that to calculate the denoiseversion and decode that calculator loss and Trace back all the way through the decoderand the process and the unit back to the latents right the reason I'm notdoing that is because that takes a lot of memory so you'll see for examplelike the clip guided diffusion notebook from the hugging face examples they do it thatway but they have to use tricks like gradient checkpointing and so on to kindof keep the RAM usage under control and for simple losses it works fine todo it this way because now we're just tracing back through denoise latents is equalto latents minus Sigma times this noise prediction right so we don't have to traceany gradients back through the unit but if you wanted to get more accurate gradientsmaybe it's not working as well as you'd hoped you can do it that otherway that I described um but however you do it very very powerful technique funto be able to again inject some additional control into this generation process by craftinga loss that expresses exactly what you'd like to see all right that's the endof The Notebook for now if you have any questions feel free to reach outto me I'll be in the forums and you can find me on Twitter andso on but for now enjoy and I can't wait to see what you make

now this stable diffusion is a latent diffusion model and这一部分是指什么意思

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

“**Stable Diffusion 是一个 latent diffusion model**” 这句话的意思是：

它**不是直接在原始像素图片上做扩散**，而是先把图片压缩到一个更小的“潜空间”（latent space）里，再在这个空间里加噪声、去噪、生成图片。

可以把它理解成：

- **像素空间**：一张 $512 \times 512$ 的图片，数据很多，很大
- **latent 空间**：把这张图压缩成更小的表示，比如 $64 \times 64$ 的特征图，信息更浓缩

所以流程大概是：

1. 真实图片先被 **VAE 编码** 成 latent
2. 扩散模型在 latent 里做 **加噪 / 去噪**
3. 最后再用 VAE **解码** 回图片

为什么要这样做？

- **更快**：处理的数值更少
- **更省显存**
- **还能保留很多图像信息**

你可以把它想成：

> 不直接在整张高清照片上改，而是先把照片“压缩成草图”，在草图上生成和修正，最后再还原成高清图。

如果你愿意，我可以接着把这句话里的 **latent**、**diffusion**、**model** 三个词分别拆开讲。

<details class='token-usage-details'><summary>$0.0154</summary>

`total=19,991 | in=19,521 | out=470 | cached=49.2% | reasoning=156 | searches=0 | $0.0154`

</details>


What's the meaning of stable

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In **Stable Diffusion**, **stable** means the model is designed to produce **reliable, high-quality results without collapsing or becoming unstable during generation**.

A simple way to think about it:

- **stable** = works well and consistently
- it doesn’t mean “never changes”
- it means the generation process is **well-behaved** and **doesn’t blow up**

So **Stable Diffusion** is basically:

> a diffusion model that has been made more stable and practical to use for image generation

If you want, I can next explain what **diffusion** means in one minute.

<details class='token-usage-details'><summary>$0.0045</summary>

`total=20,065 | in=19,894 | out=171 | cached=94.6% | reasoning=40 | searches=0 | $0.0045`

</details>


What does diffusion model include broadly? How has it developed?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Broadly, a diffusion model has a few core pieces:

- **Forward process**: slowly add noise to data until it becomes almost pure noise.
- **Reverse process**: learn how to remove that noise step by step.
- **Noise schedule**: decide how much noise is added/removed at each step.
- **Neural network**: usually predicts the noise, or predicts a related quantity.
- **Sampler**: the method used to turn those predictions into a final image.
- **Conditioning**: extra input like text, class labels, or another image.
- **Optional compression**: in **latent diffusion**, the model works in a compressed latent space instead of raw pixels.

How it developed, very roughly:

1. **Early ideas** came from score matching and stochastic processes.
2. **DDPM (2020)** made diffusion models practical and showed they could generate strong images.
3. People then improved **samplers** so generation needed fewer steps and ran faster.
4. Then came **classifier-free guidance**, which made text prompts influence results much more strongly.
5. **Latent diffusion** moved generation into compressed space, making it much cheaper and faster.
6. After that, diffusion models expanded into **text-to-image**, **image editing**, **video**, **audio**, and even **3D**.
7. More recently, a big focus has been **speed**: fewer steps, better solvers, and distillation.

So the big story is:  
**from “a clever noise-removal idea” → to “a powerful, flexible generative framework.”**

If you want, I can next give you a **one-diagram mental model** of how a diffusion model works end to end.

<details class='token-usage-details'><summary>$0.0073</summary>

`total=20,644 | in=20,091 | out=553 | cached=93.7% | reasoning=195 | searches=0 | $0.0073`

</details>
